# NLP Task 3 – Build a Chatbot using Hugging Face Transformers

## Objective
The objective of this task is to build a simple conversational chatbot using a pre-trained transformer model from Hugging Face. The chatbot takes user input, generates responses, and continues the conversation until the user types exit or quit.

In [8]:
!pip install transformers torch -q

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [10]:
model_name = "microsoft/DialoGPT-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded")

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded


In [12]:
chat_history_ids = None

print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a nice day.")
        break

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)

        if bot_input_ids.shape[-1] > 500:
            bot_input_ids = bot_input_ids[:, -500:]
    else:
        bot_input_ids = new_input_ids

    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long).to(device)

    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=40,
        top_p=0.9,
        temperature=0.7
    )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.
You: Hello
Chatbot: Hi!
You: What is Artificial Intelligence?
Chatbot: I am also a robot.
You: Who created Python?
Chatbot: What does this have to do with Python?
You: What is machine learning?
Chatbot: What is a machine learning algorithm?
You: Thank you
Chatbot: It's a machine learning algorithm.
You: exit
Chatbot: Goodbye! Have a nice day.
